# MEDIC field-map QA

Discover generated Warpkit and niimath maps, run inexpensive structural/value checks on the first, middle, and last frames, compare matching maps, and inspect the complete 4D series interactively with [ipyniivue](https://github.com/niivue/ipyniivue).

The sampled statistics are a screening tool, not a substitute for scrolling through the full image. In particular, there is no universal acceptable field-map range: acquisition and anatomy determine the plausible values.

## Setup

From the repository root, create a QA environment and launch this notebook:

```bash
python3.12 -m venv .venv
.venv/bin/python -m pip install -r requirements-qa.txt
.venv/bin/python -m jupyter lab notebooks/fieldmap_qa.ipynb
```

Run Jupyter on the same host that contains `bench_out/`; the generated images remain ignored by Git. The viewer loads one full-resolution frame at a time so the notebook does not embed the entire 4D file. Clear all outputs before committing the notebook.

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

import nibabel as nib
import numpy as np
import ipywidgets as widgets
from IPython.display import JSON, display
from ipyniivue import NiiVue, SliceType


def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "bench.py").is_file():
            return candidate
    raise FileNotFoundError("Could not find bench.py above the notebook working directory")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.fieldmap_qa import compare_maps, discover_maps, filter_maps, summarize_map

OUTPUT_ROOT = Path(os.environ.get("MEDIC_BENCH_OUTPUT_ROOT", REPO_ROOT / "bench_out")).expanduser().resolve()
records = discover_maps(OUTPUT_ROOT)
if not records:
    raise FileNotFoundError(f"No fmap NIfTI files found beneath {OUTPUT_ROOT}")

print(f"Repository: {REPO_ROOT}")
print(f"Discovered {len(records)} generated maps beneath {OUTPUT_ROOT}")

## Select a map

Edit the filters below. `None` accepts any value; when `THREADS` is `None`, the highest matching thread count is selected.

In [ ]:
MACHINE = None
DATASET = "openneuro-ds005123"
TOOL = "niimath"                 # "niimath" or "warpkit"
THREADS = None                   # None selects the largest available count
KIND = "fieldmaps"              # fieldmaps, fieldmaps_native, displacementmaps

matches = filter_maps(
    records,
    machine=MACHINE,
    dataset=DATASET,
    tool=TOOL,
    threads=THREADS,
    kind=KIND,
)
if not matches:
    available = "\n".join(f"  {index:2d}: {record.label}" for index, record in enumerate(records))
    raise ValueError(f"No map matched the filters. Available maps:\n{available}")
if THREADS is None:
    highest = max(record.threads for record in matches)
    matches = [record for record in matches if record.threads == highest]

selected = matches[0]
print(f"Selected: {selected.label}")
print(selected.path)

## Sampled structural and value QA

A `pass` means the sampled frames are finite and non-empty and the NIfTI geometry is usable. Review the robust range, zero fraction, orientation, qform/sform codes, and per-frame statistics rather than treating the status as a biological-quality verdict.

In [ ]:
summary = summarize_map(selected)
display(JSON(summary, expanded=True))

## Matching cross-tool comparison

This checks geometry and reports sampled correlation and absolute differences against the other implementation. It deliberately does not impose a numerical-agreement threshold.

In [ ]:
other_tool = "warpkit" if selected.tool == "niimath" else "niimath"
peers = filter_maps(
    records,
    machine=selected.machine,
    dataset=selected.dataset,
    tool=other_tool,
    threads=selected.threads,
    kind=selected.kind,
)
if peers:
    comparison = compare_maps(selected, peers[0])
    display(JSON(comparison, expanded=True))
else:
    print(f"No matching {other_tool} map was found for {selected.label}")

## Interactive full-resolution frame viewer

Use the crosshairs and multiplanar views to check spatial smoothness, discontinuities, edge behavior, and temporal stability. The slider can visit every volume, not only the three statistically sampled frames. Each selected frame is loaded on demand so saving the notebook cannot embed the full multi-hundred-megabyte 4D file.

In [ ]:
viewer = NiiVue(
    height=700,
    slice_type=SliceType.MULTIPLANAR,
    back_color=(0.04, 0.05, 0.07, 1),
)
viewer.opts.is_colorbar = True

source_image = nib.load(str(selected.path), mmap=True)
frame_count = source_image.shape[3] if len(source_image.shape) == 4 else 1
viewer_temp = tempfile.TemporaryDirectory(prefix="medic-fieldmap-qa-")
viewer_frame_path = Path(viewer_temp.name) / "selected-frame.nii.gz"

frame_slider = widgets.IntSlider(
    value=frame_count // 2,
    min=0,
    max=frame_count - 1,
    step=1,
    description="Frame",
    continuous_update=False,
)
colormaps = sorted(viewer.colormaps())
preferred = next(
    (name for name in ("blue2red", "coolwarm", "balance", "gray") if name in colormaps),
    colormaps[0],
)
colormap = widgets.Dropdown(options=colormaps, value=preferred, description="Colormap")
viewer_status = widgets.HTML()


def load_frame(change=None):
    frame = frame_slider.value if change is None else change["new"]
    if len(source_image.shape) == 4:
        data = np.asanyarray(source_image.dataobj[..., frame])
    else:
        data = np.asanyarray(source_image.dataobj)
    header = source_image.header.copy()
    nib.save(nib.Nifti1Image(data, source_image.affine, header), viewer_frame_path)
    old_volumes = list(viewer.volumes)
    viewer.load_volumes([{"path": str(viewer_frame_path), "colormap": colormap.value}])
    for volume in old_volumes:
        volume.close()
    viewer_status.value = f"Showing full-resolution frame {frame + 1} of {frame_count}"


def change_colormap(change):
    if viewer.volumes:
        viewer.set_colormap(viewer.volumes[0].id, change["new"])


frame_slider.observe(load_frame, names="value")
colormap.observe(change_colormap, names="value")
load_frame()

display(widgets.VBox([widgets.HBox([frame_slider, colormap]), viewer_status, viewer]))

## Optional command-line report

The same sampled checks can run without Jupyter:

```bash
.venv/bin/python scripts/fieldmap_qa.py \
  --root bench_out \
  --dataset openneuro-ds005123 \
  --kind fieldmaps \
  --json results/fieldmap-qa.json
```

The JSON report contains no image data and is safe to retain; `bench_out/` remains ignored.